In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
!pip install torchvision torch matplotlib

import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
print(torch.cuda.is_available())

True


In [16]:
import os

root = '/content/drive/MyDrive/Colab Notebooks'
data_root = os.path.join(root, 'data')
model_root = os.path.join(root, 'model')

os.environ['TORCH_HOME'] = model_root

In [17]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    # transforms.RandomHorizontalFlip(),
    # transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_dataset = datasets.OxfordIIITPet(
    root=data_root,
    target_types='category',
    transform=transform_train,
    download=True,
    )

val_dataset = datasets.OxfordIIITPet(
    root=data_root,
    split='test',
    target_types='category',
    transform=transform_val,
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [18]:
# model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
# model_name = 'vit_b_16'
# # freeze layers [:n-2]
# for name, param in model.named_parameters():
#     if 'encoder.layers.encoder_layer_10' not in name and \
#        'encoder.layers.encoder_layer_11' not in name and \
#        'heads' not in name:
#         param.requires_grad = False

# # substitude head
# model.heads.head = nn.Linear(model.heads.head.in_features, 37)
# model = model.cuda()

In [19]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model_name = 'resnet50'
# freeze layers [:n-2]
for name, param in model.named_parameters():
    if 'layer4' not in name and 'fc' not in name:
        param.requires_grad = False

# substitude head
model.fc = nn.Linear(model.fc.in_features, 37)
model = model.cuda()

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /content/drive/MyDrive/Colab Notebooks/model/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 103MB/s]


In [20]:
# fine tuning backbone
epochs = 10
best_val_acc = 0
patience = epochs
no_improve_count = 0
BEST_MODEL_PATH = os.path.join(model_root, f'{model_name}_pet_best.pth')
CKPT_PATH = os.path.join(model_root, f'{model_name}_pet_checkpoint.pth')

def get_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# load checkpoint
start_epoch = 0
if os.path.exists(CKPT_PATH):
    checkpoint = torch.load(CKPT_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']
    no_improve_count = checkpoint['no_improve_count']
    print(f"[{get_time()}] resume training from epoch {start_epoch}")
else:
    print(f"[{get_time()}] start training")

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.cuda(), labels.cuda()
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.cuda(), labels.cuda()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

for epoch in range(start_epoch, epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    scheduler.step()
    print(f"[{get_time()}] Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        no_improve_count = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_acc': val_acc,
            'best_val_acc': best_val_acc,
            'no_improve_count': no_improve_count,
        }, BEST_MODEL_PATH)
        print(f"[{get_time()}] Best model saved, Val Acc={best_val_acc:.4f}")
    else:
        no_improve_count += 1
        print(f"[{get_time()}] No improvement ({no_improve_count}/{patience})")

    if no_improve_count >= patience:
        print(f"[{get_time()}] Early Stop Val Acc={best_val_acc:.4f}")
        break

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_acc': val_acc,
        'best_val_acc': best_val_acc,
        'no_improve_count': no_improve_count,
    }, CKPT_PATH)



[2026-06-09 07:24:50] start training
[2026-06-09 07:26:00] Epoch 1: Train Acc=0.7527, Val Acc=0.8746
[2026-06-09 07:26:00] Best model saved, Val Acc=0.8746
[2026-06-09 07:27:09] Epoch 2: Train Acc=0.9516, Val Acc=0.8942
[2026-06-09 07:27:09] Best model saved, Val Acc=0.8942
[2026-06-09 07:28:21] Epoch 3: Train Acc=0.9821, Val Acc=0.9081
[2026-06-09 07:28:22] Best model saved, Val Acc=0.9081
[2026-06-09 07:29:33] Epoch 4: Train Acc=0.9918, Val Acc=0.9030
[2026-06-09 07:29:33] No improvement (1/10)
[2026-06-09 07:30:40] Epoch 5: Train Acc=0.9951, Val Acc=0.9032
[2026-06-09 07:30:40] No improvement (2/10)
[2026-06-09 07:31:47] Epoch 6: Train Acc=0.9986, Val Acc=0.9109
[2026-06-09 07:31:48] Best model saved, Val Acc=0.9109
[2026-06-09 07:33:00] Epoch 7: Train Acc=0.9995, Val Acc=0.9147
[2026-06-09 07:33:01] Best model saved, Val Acc=0.9147
[2026-06-09 07:34:13] Epoch 8: Train Acc=1.0000, Val Acc=0.9120
[2026-06-09 07:34:13] No improvement (1/10)
[2026-06-09 07:35:19] Epoch 9: Train Acc=1.0